## **IN4640 Assignment 2**

## **Question 01**

### **Part (a)**

**Total Least Squares fitting using only first line points**

I use only the first line, meaning columns x1 and y1.

Total Least Squares minimizes the perpendicular distance to the line.
I use SVD to compute the best fitting line.

The general line equation is **ax + by + c = 0**

**For Total Least Squares:**

01. Take the points (x1, y1)

02. Compute centroid

03. Subtract centroid

04. Apply SVD

05. The line direction is the singular vector corresponding to the smallest singular value

06. Convert to line parameters

In [1]:
import numpy as np
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
import random
import cv2

In [2]:
# Load data
D = np.genfromtxt("lines.csv", delimiter=",", skip_header=1)

# First line points
x1 = D[:, 0]
y1 = D[:, 3]

points = np.column_stack((x1, y1))

# Compute centroid
centroid = np.mean(points, axis=0)

# Subtract centroid
points_centered = points - centroid

# SVD
U, S, Vt = np.linalg.svd(points_centered)

# Line direction = last singular vector
direction = Vt[-1]

a = direction[0]
b = direction[1]

# Compute c using centroid
c = - (a * centroid[0] + b * centroid[1])

print("Total Least Squares Line Parameters")
print("a =", a)
print("b =", b)
print("c =", c)

Total Least Squares Line Parameters
a = 0.7735616496467872
b = -0.6337210539312553
c = -3.794192210845812


### **Part (b)**

RANSAC to detect three lines

now use all points

x1, x2, x3
y1, y2, y3

RANSAC procedure:

01. Randomly sample two points

02. Fit a line

03. Compute distances of all points to the line

04. Count inliers under a threshold

05. Keep the model with highest consensus

06. Remove inliers

07. Repeat to find remaining lines

In [3]:
# Prepare all points
X_cols = D[:, :3]
Y_cols = D[:, 3:]

X_all = X_cols.flatten()
Y_all = Y_cols.flatten()

points = np.column_stack((X_all, Y_all))

def fit_line(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    
    a = y1 - y2
    b = x2 - x1
    c = x1*y2 - x2*y1
    
    norm = np.sqrt(a*a + b*b)
    return a/norm, b/norm, c/norm

def distance(a, b, c, x, y):
    return abs(a*x + b*y + c)

def ransac(points, threshold=0.1, iterations=1000):
    best_inliers = []
    best_model = None
    
    for _ in range(iterations):
        idx = np.random.choice(len(points), 2, replace=False)
        p1, p2 = points[idx]
        
        a, b, c = fit_line(p1, p2)
        
        inliers = []
        for i, (x, y) in enumerate(points):
            if distance(a, b, c, x, y) < threshold:
                inliers.append(i)
        
        if len(inliers) > len(best_inliers):
            best_inliers = inliers
            best_model = (a, b, c)
    
    return best_model, best_inliers

remaining_points = points.copy()
models = []

for i in range(3):
    model, inliers = ransac(remaining_points)
    models.append(model)
    
    remaining_points = np.delete(remaining_points, inliers, axis=0)

for i, (a, b, c) in enumerate(models):
    print(f"Line {i+1} parameters:")
    print("a =", a)
    print("b =", b)
    print("c =", c)
    print()

Line 1 parameters:
a = 0.8298741736332359
b = -0.5579505855687883
c = -3.7121704170094674

Line 2 parameters:
a = -0.36315076431795185
b = -0.9317303914627274
c = 2.02172823114128

Line 3 parameters:
a = 0.7294977387738967
b = -0.6839832228379374
c = 0.6349408598970802



**Visualization**

In [41]:
plt.figure(figsize=(6,6))

colors = ['r','g','b']

for i, (a, b, c) in enumerate(models):
    xs = np.linspace(min(X_all), max(X_all), 100)
    ys = (-a*xs - c) / b
    plt.plot(xs, ys, colors[i])

plt.scatter(X_all, Y_all, s=10)
plt.title("RANSAC Line Fitting")
plt.show()

RANSAC Line Fitting Result

The plot shows the result of applying RANSAC to detect three lines from the dataset.

Blue points represent all input data points.
Each colored line represents one line detected by RANSAC.

RANSAC works by:
Randomly selecting two points
Estimating a line model
Counting inliers within a distance threshold
Selecting the model with maximum consensus
Removing inliers and repeating the process

The algorithm successfully separates the three linear structures despite noise and mixed data.

This demonstrates that RANSAC is robust to outliers and effective for multi-model fitting problems.

## **Question 02**

**Interactive Pixel Measurement**

In [42]:
image_path = "earrings.jpg"

img = cv2.imread(image_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

plt.imshow(gray, cmap='gray')
plt.title("Grayscale")
plt.show()

In [43]:
# Blur to reduce noise
blur = cv2.GaussianBlur(gray, (9,9), 2)

# Hough Circle Detection
circles = cv2.HoughCircles(
    blur,
    cv2.HOUGH_GRADIENT,
    dp=1.2,
    minDist=200,
    param1=100,
    param2=30,
    minRadius=100,
    maxRadius=400
)

if circles is not None:
    circles = np.uint16(np.around(circles))
    for i in circles[0,:1]:   
        center = (i[0], i[1])
        radius = i[2]
        
        cv2.circle(img, center, radius, (0,255,0), 3)
        cv2.circle(img, center, 2, (0,0,255), 3)

plt.figure(figsize=(6,6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Detected Circle")
plt.show()

pixel_diameter = 2 * radius
print("Pixel Diameter:", pixel_diameter)

Pixel Diameter: 382


f = 8 mm

Z = 720 mm

pixel size = 2.2 μm = 2.2e-3 mm

**Camera Projection Model**

Using the pinhole camera model:

x = (f / Z) X

Rearranging to compute real size:

X = (Z / f) × x

Since image measurement is in pixels:

x = (pixel diameter) × (pixel size)

Therefore,

Real Size = (Z / f) × (pixel diameter × pixel size)

In [7]:
f = 8
Z = 720
pixel_size = 2.2e-3

real_size = (Z / f) * (pixel_diameter * pixel_size)

print("Estimated Real Diameter (mm):", real_size)

Estimated Real Diameter (mm): 75.63600000000001


The estimated real diameter of each earring is approximately **75.6 mm**.

## **Question 03**

**Objective**

Select four corner points on the cricket turf and superimpose a country flag using homography transformation.

**Method**

01. Select four points on the turf image
02. Define four corner points of the flag image
03. Compute homography matrix
04. Warp the flag image
05. Blend warped flag onto turf

This uses perspective transformation based on projective geometry.

In [34]:
turf = cv2.imread("turf.jpg")
if turf is None:
    raise FileNotFoundError("jpg not found")

turf_rgb = cv2.cvtColor(turf, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(turf_rgb)
ax.set_title("Click 4 turf corner points CLOCKWISE (top-left → top-right → bottom-right → bottom-left)", fontsize=11)
plt.tight_layout()

points = plt.ginput(4, timeout=60)
plt.close()

turf_points = np.array(points, dtype=np.float32)
print("Selected Turf Points:")
print(turf_points)

# Visualize selected points
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(turf_rgb)
labels = ['TL', 'TR', 'BR', 'BL']
colors = ['red', 'green', 'blue', 'orange']
for i, (pt, label, color) in enumerate(zip(turf_points, labels, colors)):
    ax.plot(pt[0], pt[1], 'o', color=color, markersize=10)
    ax.text(pt[0]+5, pt[1]-10, f"{i+1}. {label}", color=color, fontsize=12, fontweight='bold')
ax.set_title("Selected Points on Turf")
ax.axis('off')
plt.tight_layout()
plt.show()

Selected Turf Points:
[[243.70517 188.21115]
 [465.5698  191.20932]
 [350.14023 309.63705]
 [132.77286 296.1453 ]]


In [35]:
flag = cv2.imread("flag.png")
if flag is None:
    raise FileNotFoundError("flag.png.")

flag_rgb = cv2.cvtColor(flag, cv2.COLOR_BGR2RGB)

h, w, _ = flag.shape
print(f"Flag loaded: {w}x{h} pixels")

flag_points = np.array([
    [0, 0],    # top-left
    [w, 0],    # top-right
    [w, h],    # bottom-right
    [0, h]     # bottom-left
], dtype=np.float32)

plt.figure(figsize=(6, 4))
plt.imshow(flag_rgb)
plt.title("Flag to Superimpose")
plt.axis('off')
plt.show()

Flag loaded: 1280x640 pixels


In [44]:
H, status = cv2.findHomography(flag_points, turf_points, cv2.RANSAC, 5.0)

warped_flag = cv2.warpPerspective(
    flag_rgb,
    H,
    (turf_rgb.shape[1], turf_rgb.shape[0])
)

print("Homography computed and flag warped successfully")
print(f"Homography Matrix:\n{H}")

Homography computed and flag warped successfully
Homography Matrix:
[[ 1.42132962e-01 -1.78489143e-01  2.43705170e+02]
 [-1.04710045e-02  1.57143683e-01  1.88211151e+02]
 [-6.70120262e-05 -3.88438229e-05  1.00000000e+00]]


In [45]:
mask = np.zeros_like(turf_rgb)
cv2.fillConvexPoly(mask, turf_points.astype(np.int32), (255, 255, 255))

mask_inv = cv2.bitwise_not(mask)

turf_bg = cv2.bitwise_and(turf_rgb, mask_inv)         
flag_fg = cv2.bitwise_and(warped_flag, mask)        

flag_alpha = 0.75   

result = cv2.addWeighted(turf_bg, 1.0, flag_fg, flag_alpha, 0)

mask_blur = cv2.GaussianBlur(mask.astype(np.float32) / 255.0, (21, 21), 0)
result_soft = (turf_rgb * (1 - mask_blur * flag_alpha) + warped_flag * (mask_blur * flag_alpha)).astype(np.uint8)

print("Blending complete")

Blending complete


In [47]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(turf_rgb)
axes[0].set_title("Original Turf", fontsize=13)
axes[0].axis('off')

axes[1].imshow(result)
axes[1].set_title(f"Flag Superimposed\n(Hard mask, α={flag_alpha})", fontsize=13)
axes[1].axis('off')

axes[2].imshow(result_soft)
axes[2].set_title("Flag Superimposed\n(Soft edge blend)", fontsize=13)
axes[2].axis('off')

plt.suptitle("Cricket Turf - Flag Superimposition", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("result_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: result_comparison.png")

Saved: result_comparison.png


In [48]:
plt.figure(figsize=(12, 8))
plt.imshow(result_soft)
plt.title("Superimpose the a country flag on the turf.", fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig("final_result.png", dpi=200, bbox_inches='tight')
plt.show()
print("Saved: final_result.png")

Saved: final_result.png


In [49]:
flag_alpha_custom = 0.65 

result_custom = (turf_rgb * (1 - mask_blur * flag_alpha_custom) + 
                 warped_flag * (mask_blur * flag_alpha_custom)).astype(np.uint8)

plt.figure(figsize=(10, 7))
plt.imshow(result_custom)
plt.title(f"Custom Alpha = {flag_alpha_custom}")
plt.axis('off')
plt.show()